In [1]:
import json
import time
import requests
import urllib.parse
from tqdm.auto import tqdm
from datasets import load_dataset

# -------------------------
# LOAD DATASET
# -------------------------
ds = load_dataset("MinaGabriel/entityquestions-retrieval-top20-all")["train"]

import json

pop_dict = {}

with open("s_pop_results_fixed.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        pop_dict[row["subject"]] = {
            "s_pop_sum": row["s_pop_sum"],
            "s_pop_avg": row["s_pop_avg"]
        }

In [2]:
print(pop_dict["Ilango Adigal"])

{'s_pop_sum': 32637, 's_pop_avg': 2719}


In [22]:
def transform_example(ex):
    # -------------------------
    # 1. Rename subject → subj
    # -------------------------
    subj = ex["subject"]

    # -------------------------
    # 2. Attach s_pop
    # -------------------------
    s_pop = pop_dict.get(subj, 0)

    # -------------------------
    # 3. Extract retrieved_docs TEXT ONLY
    # -------------------------
    retrieved_texts = []

    for doc in ex.get("retrieved_docs", []):
        if isinstance(doc, dict):
            retrieved_texts.append(doc.get("text", ""))
        else:
            retrieved_texts.append(doc)

    # -------------------------
    # 4. Build new clean record
    # -------------------------
    return {
        "subj": subj,
        "prop": ex.get("relation", None), 

        "s_pop": s_pop["s_pop_sum"],
        "s_pop_sum": s_pop["s_pop_sum"],
        "s_pop_avg": s_pop["s_pop_avg"],
        "question": ex.get("question", None),
        "possible_answers": ex.get("answers", []),

        "retrieved_docs": retrieved_texts
    }

In [23]:
from datasets import load_dataset

ds = load_dataset("MinaGabriel/entityquestions-retrieval-top20-all")["train"]

ds_clean = ds.map(transform_example, remove_columns=ds.column_names)

Map:   0%|          | 0/22075 [00:00<?, ? examples/s]

In [24]:
print(json.dumps(ds_clean[0], indent=2, ensure_ascii=False))


{
  "question": "What kind of work does Joe Bianchi do?",
  "retrieved_docs": [
    "\"Joe Bianchi\"\nJoe Bianchi Joe Bianchi (August 5, 1871 in Origgio, Italy – May 29, 1949) was a world-class spur maker who immigrated to Victoria, Texas in December 1885 and died in Victoria at the age of 77. Bianchi's particular style became known across Southwest Texas as the \"\"Victoria Shank\"\" or \"\"bottle-opener\"\" spur, terms which are still used today by collectors, some sixty years after production of these much sought-after custom, handmade spurs ceased. Bianchi's father, Luigi (Louis), purchased a farm after arriving in Texas. The 1900 Census listed Louis and his youngest son as \"\"farmer\"\". By contrast, Joe and his older",
    "\"Joe Bianchi\"\nthe local Spanish American militia unit and was an active member of the Victoria Fire Department right up to the final years of his life. For his industriousness, civic mindedness and decent living, Joe was highly respected by all who knew an

In [25]:
ds_clean.push_to_hub("MinaGabriel/entityquestions-final-clean")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/MinaGabriel/entityquestions-final-clean/commit/14271e84f89e246912242c071328ad084db57658', commit_message='Upload dataset', commit_description='', oid='14271e84f89e246912242c071328ad084db57658', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MinaGabriel/entityquestions-final-clean', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MinaGabriel/entityquestions-final-clean'), pr_revision=None, pr_num=None)

In [ ]:
import json
import time
import requests
import urllib.parse
from tqdm.auto import tqdm
from datasets import load_dataset

# -------------------------
# LOAD DATASET
# -------------------------
ds = load_dataset("MinaGabriel/entityquestions-retrieval-top20-all")["train"]

# -------------------------
# SETTINGS
# -------------------------
OUTFILE = "s_pop_results.jsonl"
START_IDX = 0    
YEAR = 2023

# -------------------------
# CACHE (avoid duplicate API calls)
# -------------------------
_pop_cache = {}

# -------------------------
# WIKIPEDIA API FUNCTION
# -------------------------
def get_s_pop(title, retries=3, sleep_time=1):
    if title in _pop_cache:
        return _pop_cache[title]

    title_clean = urllib.parse.quote(title.replace(" ", "_"), safe="_()")

    url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/{title_clean}/monthly/{YEAR}0101/{YEAR}1231"

    headers = {
        "User-Agent": "Mina-FUSEQA-Research/1.0 (mina@example.com)"
    }

    for _ in range(retries):
        try:
            res = requests.get(url, headers=headers, timeout=5)

            if res.status_code == 200:
                data = res.json()
                items = data.get("items", [])

                if not items:
                    result = {"sum": 0, "avg": 0}
                else:
                    views = [x.get("views", 0) for x in items]
                    total = int(sum(views))
                    avg = int(total / len(views))

                    result = {
                        "sum": total,
                        "avg": avg
                    }

                _pop_cache[title] = result
                return result

            elif res.status_code >= 500:
                time.sleep(sleep_time)
                continue
            else:
                break

        except:
            time.sleep(sleep_time)

    result = {"sum": 0, "avg": 0}
    _pop_cache[title] = result
    return result


# -------------------------
# RUN
# -------------------------
with open(OUTFILE, "a", encoding="utf-8") as f:

    for i in tqdm(range(START_IDX, len(ds))):

        ex = ds[i]
        subject = ex["subject"]

        pop = get_s_pop(subject)

        record = {
            "id": i,
            "subject": subject,
            "s_pop_sum": pop["sum"],
            "s_pop_avg": pop["avg"]
        }

        f.write(json.dumps(record) + "\n")
        f.flush()   

In [ ]:
import json
import time
import requests
import urllib.parse
from tqdm.auto import tqdm

INFILE  = "s_pop_results.jsonl"
OUTFILE = "s_pop_results_fixed.jsonl"
YEAR = 2023

# -------------------------
# API FUNCTION (same as before, slightly stronger)
# -------------------------
def get_s_pop(title, retries=5, sleep_time=2):
    title_clean = urllib.parse.quote(title.replace(" ", "_"), safe="_()")

    url = f"https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/en.wikipedia/all-access/user/{title_clean}/monthly/{YEAR}0101/{YEAR}1231"

    headers = {
        "User-Agent": "Mina-FUSEQA-Research/1.0 (mina@example.com)"
    }

    for _ in range(retries):
        try:
            res = requests.get(url, headers=headers, timeout=10)

            if res.status_code == 200:
                data = res.json()
                items = data.get("items", [])

                if not items:
                    return {"sum": 0, "avg": 0}

                views = [x.get("views", 0) for x in items]
                total = int(sum(views))
                avg = int(total / len(views))

                return {"sum": total, "avg": avg}

            elif res.status_code in [429, 500, 502, 503]:
                time.sleep(sleep_time)
                continue

            else:
                return {"sum": 0, "avg": 0}

        except:
            time.sleep(sleep_time)

    return {"sum": 0, "avg": 0}


# -------------------------
# FIX LOOP
# -------------------------
with open(INFILE, "r", encoding="utf-8") as fin, \
     open(OUTFILE, "w", encoding="utf-8") as fout:

    for line in tqdm(fin):
        row = json.loads(line)
 
        if row["s_pop_sum"] == 0:
            pop = get_s_pop(row["subject"])
            row["s_pop_sum"] = pop["sum"]
            row["s_pop_avg"] = pop["avg"]

            time.sleep(0.5)  

        fout.write(json.dumps(row) + "\n")